# Function Description

Create a function that traverses through a selected folder and its subfolders, and finds all files with specific extensions. The extensions to search for are `.csproj` and `.sln`.

## Function Requirements

1. **File Search**:
    - Traverse through the selected folder and its subfolders.
    - Find all files with the extensions `.csproj` and `.sln`.

2. **Package References**:
    - Find all package references in the found files.
    - For each package reference, run `dotnet nuget why <PROJECT|SOLUTION> <PACKAGE> [-f|--framework <FRAMEWORK>]` on each of the found files.
    - Return the results in a structured format.

3. **Error Handling**:
    - Handle errors gracefully.
    - Provide meaningful error messages.

4. **Performance**:
    - Handle large numbers of files efficiently.
    - Run in parallel to improve performance.

5. **Cross-Platform**:
    - Run on both Windows and Linux.

6. **Polyglot Notebook Environment**:
    - Run in a polyglot notebook environment.
    - Use the `dotnet` command line tool.

In [ ]:
function Invoke-NugetWhy {
    [CmdletBinding()]
    param (
        [Parameter(Mandatory)]
        [string]$RootFolder,

        [Parameter(Mandatory=$false)]
        [string]$Framework
    )

    if (-not (Test-Path $RootFolder)) {
        throw "Root folder '$RootFolder' does not exist."
    }

    # Holds file and package details for processing
    $packagePairs = @()
    $solutions = @()

    # Get all *.csproj and *.sln files recursively.
    $files = Get-ChildItem -Path $RootFolder -Recurse -Include *.csproj, *.sln -File

    foreach ($file in $files) {
        if ($file.Extension -ieq ".csproj") {
            try {
                # Load the file as XML and get PackageReference elements
                [xml]$xmlDoc = Get-Content $file.FullName -Raw
                # Select all PackageReference elements and get their Include attribute
                $packages = $xmlDoc.SelectNodes("//PackageReference") | ForEach-Object { $_.Include } | Where-Object { $_ }
                $packages = $packages | Select-Object -Unique
                foreach ($pkg in $packages) {
                    $packagePairs += [PSCustomObject]@{
                        File       = $file.FullName
                        Package    = $pkg
                        Identifier = "PROJECT"
                    }
                    Write-Output "Found package reference '$pkg' in file '$($file.FullName)'"
                }
            }
            catch {
                Write-Warning "Error parsing csproj file '$($file.FullName)': $($_.Exception.Message)"
                # Record error result if needed.
                $packagePairs += [PSCustomObject]@{
                    File       = $file.FullName
                    Package    = ""
                    Identifier = "PROJECT"
                    Error      = "Error parsing csproj file: $($_.Exception.Message)"
                }
            }
        }
        elseif ($file.Extension -ieq ".sln") {
            # For .sln files, package references are typically not directly defined.
            # You could extend this logic to locate associated csproj files.
            # For now, we simply ignore them.

            $solutions += [PSCustomObject]@{
                File       = $file.FullName
                Package    = ""
                Identifier = "SOLUTION"
                Error      = "Solution files do not contain package references directly."
            }
            Write-Output "Found solution file '$($file.FullName)' but no package references are directly defined."
            continue
        }
    }

    if ($packagePairs.Count -eq 0) {
        Write-Output "No package references found."
        return @()
    }

    if( $solutions.Count -gt 0) {
        Write-Output "Found $($solutions.Count) solution files. Listed: $($solutions | Select-Object -ExpandProperty File | Out-String)"
    }
    else {
        Write-Output "No solution files found."
    }

    # Filter out empty package references
    # $packagePairs = $packagePairs | Where-Object { $_.Package -ne "" } | Select-Object -Unique

    # Sort and remove duplicates
    $packagePairs = $packagePairs | Sort-Object -Property Package, File -Unique

    # Process each package reference while directly referencing $Framework
    Write-Output "Found $($packagePairs.Count) package references. Listed: $($packagePairs | Select-Object -ExpandProperty Package | Out-String)"
    
    $selectedSln = $solutions | Where-Object { $_.File -like "*$($RootFolder)*" } | Select-Object -First 1
    if ($selectedSln) {
        Write-Output "Selected solution file: $($selectedSln.File)"
    } else {
        Write-Output "No solution file selected."
    }


    # Consolidate results of dotnet nuget why into a single object for each package reference find any unused packages
    $results = @()

    # Further processing to find any unused packages can be added here
    # This may involve comparing the package references to the project dependencies
    # and determining if they are actually used in the code.
    foreach ($pair in $packagePairs) {
        $package = $pair.Package
        $file = $pair.File

        # Run the dotnet nuget why command for each package reference
        try {
            Write-Output "Running 'dotnet nuget why' for package '$package' in file '$file'"
            $whyResult = dotnet nuget why $selectedSln.File $package --framework $Framework 2>&1
            if ($whyResult) {
                $whyResult = $whyResult | Out-String
                Write-Output "dotnet nuget why result: $whyResult"
                $results += [PSCustomObject]@{
                    Package    = $package
                    File       = $file
                    Identifier = "PROJECT"
                    Result     = $whyResult
                    IsSuccess  = $true
                }
            } else {
                Write-Output "No result from 'dotnet nuget why' for package '$package' in file '$file'"
                $results += [PSCustomObject]@{
                    Package    = $package
                    File       = $file
                    Identifier = "PROJECT"
                    Result     = "No result from 'dotnet nuget why'"
                    IsSuccess  = $false
                }
            }
        } catch {
            Write-Warning "Error running 'dotnet nuget why' for package '$package' in file '$file': $($_.Exception.Message)"
            $results += [PSCustomObject]@{
                Package    = $package
                File       = $file
                Identifier = "PROJECT"
                Result     = "Error running 'dotnet nuget why': $($_.Exception.Message)"
                IsSuccess  = $false
            }
        }
    }

    return $results
}

# Usage example:
# $results = Invoke-NugetWhy -RootFolder "C:\xsource\Homemade-Cookies\Aspiring" -Framework "net9.0" | Format-List
# $results | Where-Object { $_.IsSuccess -eq $false } | Format-List
# $results | Where-Object { $_.IsSuccess -eq $true } | Format-List

$results = Invoke-NugetWhy -RootFolder "C:\xsource\Homemade-Cookies\Aspiring" -Framework "net9.0"
Write-Output $results | Format-List

Found package reference 'OpenAI' in file 'C:\xsource\Homemade-Cookies\Aspiring\Aspire.ChatApp\Aspire.ChatApp.csproj'
Found package reference 'Microsoft.Extensions.AI.OpenAI' in file 'C:\xsource\Homemade-Cookies\Aspiring\Aspire.ChatApp\Aspire.ChatApp.csproj'
Found package reference 'Microsoft.EntityFrameworkCore.Sqlite' in file 'C:\xsource\Homemade-Cookies\Aspiring\Aspire.ChatApp\Aspire.ChatApp.csproj'
Found package reference 'Microsoft.Extensions.AI' in file 'C:\xsource\Homemade-Cookies\Aspiring\Aspire.ChatApp\Aspire.ChatApp.csproj'
Found package reference 'Microsoft.SemanticKernel.Core' in file 'C:\xsource\Homemade-Cookies\Aspiring\Aspire.ChatApp\Aspire.ChatApp.csproj'
Found package reference 'PdfPig' in file 'C:\xsource\Homemade-Cookies\Aspiring\Aspire.ChatApp\Aspire.ChatApp.csproj'
Found package reference 'System.Linq.Async' in file 'C:\xsource\Homemade-Cookies\Aspiring\Aspire.ChatApp\Aspire.ChatApp.csproj'
Found package reference 'Azure.Identity' in file 'C:\xsource\Homemade-Cookie